In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt


In [2]:
path = "../.data/downscaling_splits/"
filename1 = "train.nc"
filename2 = "val.nc"
filename3 = "test.nc"

ds_train = xr.open_dataset(os.path.join(path, filename1))
ds_val   = xr.open_dataset(os.path.join(path, filename2))
ds_test  = xr.open_dataset(os.path.join(path, filename3))


In [3]:
def add_interp_residual_and_valid_mask(ds):
    # interpolate low-res Tmax to the 4 km grid
    tmax_lowres_interp = ds["tmax_lowres"].interp(
        lat_coarse=ds["lat"],
        lon_coarse=ds["lon"],
        method="linear"
    ).rename("tmax_lowres_interp")

    # residual target
    tmax_residual = (ds["tmax_highres"] - tmax_lowres_interp).rename("tmax_residual")

    # stricter valid mask: must already be valid in split file,
    # and all relevant dynamic fields must be finite for all times
    valid_mask = (
        (ds["valid_mask"] == 1) &
        ds["tmax_highres"].notnull().all(dim="time") &
        tmax_lowres_interp.notnull().all(dim="time") &
        tmax_residual.notnull().all(dim="time")
    ).astype("int8").rename("valid_mask")

    ds_out = ds.copy()
    ds_out["tmax_lowres_interp"] = tmax_lowres_interp.astype("float32")
    ds_out["tmax_residual"] = tmax_residual.astype("float32")
    ds_out["valid_mask"] = valid_mask

    return ds_out

In [4]:
ds_train = add_interp_residual_and_valid_mask(ds_train)
ds_val   = add_interp_residual_and_valid_mask(ds_val)
ds_test  = add_interp_residual_and_valid_mask(ds_test)

In [5]:
mask = ds_train["valid_mask"] == 1

# dynamic input: interpolated low-res Tmax
x_train_valid = ds_train["tmax_lowres_interp"].where(mask)
x_mean = float(x_train_valid.mean().values)
x_std  = float(x_train_valid.std().values)

# target: residual
y_train_valid = ds_train["tmax_residual"].where(mask)
y_mean = float(y_train_valid.mean().values)
y_std  = float(y_train_valid.std().values)

print("x_mean, x_std =", x_mean, x_std)
print("y_mean, y_std =", y_mean, y_std)

x_mean, x_std = 298.5472412109375 4.390291690826416
y_mean, y_std = 0.8633578419685364 1.7342606782913208


In [6]:
def normalize_split(ds, x_mean, x_std, y_mean, y_std):
    ds = ds.copy()

    ds["x_norm"] = ((ds["tmax_lowres_interp"] - x_mean) / x_std).astype("float32")
    ds["y_norm"] = ((ds["tmax_residual"] - y_mean) / y_std).astype("float32")

    return ds

ds_train_norm = normalize_split(ds_train, x_mean, x_std, y_mean, y_std)
ds_val_norm   = normalize_split(ds_val,   x_mean, x_std, y_mean, y_std)
ds_test_norm  = normalize_split(ds_test,  x_mean, x_std, y_mean, y_std)

In [7]:
def fill_invalid_with_zero(ds):
    mask = ds["valid_mask"] == 1
    ds = ds.copy()

    ds["x_norm"] = ds["x_norm"].where(mask, 0.0).fillna(0.0)
    ds["y_norm"] = ds["y_norm"].where(mask, 0.0).fillna(0.0)

    return ds

ds_train_norm = fill_invalid_with_zero(ds_train_norm)
ds_val_norm   = fill_invalid_with_zero(ds_val_norm)
ds_test_norm  = fill_invalid_with_zero(ds_test_norm)

In [8]:
for ds_name, ds in [("train", ds_train_norm), ("val", ds_val_norm), ("test", ds_test_norm)]:
    print(ds_name)
    print("  x_norm has NaN:", bool(ds["x_norm"].isnull().any()))
    print("  y_norm has NaN:", bool(ds["y_norm"].isnull().any()))

train
  x_norm has NaN: False
  y_norm has NaN: False
val
  x_norm has NaN: False
  y_norm has NaN: False
test
  x_norm has NaN: False
  y_norm has NaN: False


In [9]:
ds_train_norm.to_netcdf("../.data/downscaling_splits/train_norm.nc")
ds_val_norm.to_netcdf("../.data/downscaling_splits/val_norm.nc")
ds_test_norm.to_netcdf("../.data/downscaling_splits/test_norm.nc")

In [10]:
norm_stats = xr.Dataset(
    {
        "x_mean": xr.DataArray(x_mean),
        "x_std": xr.DataArray(x_std),
        "y_mean": xr.DataArray(y_mean),
        "y_std": xr.DataArray(y_std),
    }
)

norm_stats.to_netcdf("../.data/downscaling_splits/norm_stats.nc")